# Original PCBert-Kla training to KlaDB external study

This experiment compares PCBert-Kla (`baseline`) and MM-Kla (`token_gated`) using the same original 3,487-row PCBert-Kla training file. A candidate-protein/homology-group-disjoint validation subset is formed **within that file** to select the training epoch by AUPRC. Each architecture is then initialized again, trained on **all 3,487 original rows** for its selected number of epochs, and evaluated once on the locked, separate KlaDB human study (PMID 41192556). This run does not use the external study for fitting, epoch selection, or threshold tuning.

The original training data contain repeated windows and candidate rather than verified parent-protein identities. External negative labels mean unreported lysines, not experimentally established non-lactylated sites. This same KlaDB holdout has been examined in earlier project experiments, so it is **not a newly blinded test**. The result is an external-study evaluation of the stated protocol, not proof that the original training source is independently verified.


## 1. Runtime and run control

Select a Kaggle **2 x T4 GPU** runtime with Internet enabled. Attach the private Kaggle dataset containing `kladb_human_external_test.csv` and `external_test_manifest.json`, and create a Kaggle secret named `GITHUB_TOKEN` with read access to the private repository. Training is disabled until `RUN_TRAINING` is set to `True`. One seed can be run per saved version; multiple seeds can also be listed in `RUN_SEEDS`.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile, time

REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
REPO_DIR = Path('/kaggle/working/MM-Kla')
PROJECT = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT = Path('/kaggle/working/original_train_kladb_external_results')
RUN_TRAINING = False
RUN_SEEDS = (42,)
DECLARED_SEEDS = (42, 123, 777)
EPOCHS = 30
BATCH_SIZE = 4
if not RUN_SEEDS or len(set(RUN_SEEDS)) != len(RUN_SEEDS) or any(seed not in DECLARED_SEEDS for seed in RUN_SEEDS):
    raise ValueError('RUN_SEEDS must be distinct members of DECLARED_SEEDS')
OUTPUT.mkdir(parents=True, exist_ok=True)


## 2. Source checkout and dependencies

Only the tracked original training files are obtained from GitHub. The access token is passed to Git through a temporary askpass helper; it is not printed or stored in notebook outputs. The checked-out commit and GPU hardware are recorded alongside the results.


In [ ]:
if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" x-access-token ;;\n  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'MM_KLA_GITHUB_TOKEN': token})
        result = subprocess.run(['git', '-c', 'credential.helper=', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(REPO_DIR)], env=env, capture_output=True, text=True)
        del token, env
    if result.returncode:
        raise RuntimeError('Private clone failed; check repository access and Kaggle Internet')
if not (PROJECT / 'scripts/run_original_kladb_external.py').is_file():
    raise RuntimeError('The checked-out repository does not contain this experiment runner; update the checkout')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle GPU runtime is required')
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
runtime = {'git_commit': COMMIT, 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'torch': torch.__version__, 'seeds': RUN_SEEDS, 'max_epochs': EPOCHS, 'batch_size': BATCH_SIZE}
(OUTPUT / 'runtime.json').write_text(json.dumps(runtime, indent=2) + '\n')
print(json.dumps(runtime, indent=2))


## 3. External dataset and preflight

The attached dataset must contain exactly one matching pair of export files. The runner checks its checksum and audited counts, the original source-file checksums, exact training/test window overlap, feature schema, and grouped validation separation. This cell performs **no model training and no external inference**.


In [ ]:
matches = sorted(Path('/kaggle/input').rglob('kladb_human_external_test.csv'))
if len(matches) != 1:
    raise RuntimeError(f'Attach exactly one private KlaDB external dataset; found {len(matches)} CSV files')
EXTERNAL_CSV = matches[0]
EXTERNAL_MANIFEST = EXTERNAL_CSV.with_name('external_test_manifest.json')
if not EXTERNAL_MANIFEST.is_file():
    raise FileNotFoundError(EXTERNAL_MANIFEST)
manifest = json.loads(EXTERNAL_MANIFEST.read_text())
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
if manifest['sha256'] != sha256(EXTERNAL_CSV):
    raise ValueError('Attached external CSV does not match its manifest')
print('External dataset:', EXTERNAL_CSV.parent)
print('External CSV SHA-256:', manifest['sha256'])
preflight = subprocess.run([sys.executable, 'scripts/run_original_kladb_external.py', '--architecture', 'baseline', '--seed', str(RUN_SEEDS[0]), '--external-csv', str(EXTERNAL_CSV), '--external-manifest', str(EXTERNAL_MANIFEST), '--output-dir', str(OUTPUT / 'preflight_unused'), '--check-only'], cwd=PROJECT, text=True, capture_output=True)
if preflight.returncode:
    raise RuntimeError('Preflight failed:\n' + preflight.stdout[-3000:] + preflight.stderr[-3000:])
print(preflight.stdout)


## 4. Matched two-GPU training

For each selected seed, the two architectures run in separate processes, one per T4. Both use ProtBert (`Rostlab/prot_bert`, first four encoder layers fine-tuned), 27 sequence-derived physicochemical features, batch size 4, SGD with learning rate 0.003, no positive weighting, no scheduler, and a fixed 0.5 decision threshold. Each run saves grouped-validation history, a full-data refit checkpoint and scaler, external probabilities, final metrics, and its console log. Model selection uses validation AUPRC only.

Kaggle Save Version starts a fresh `/kaggle/working`; collect output bundles across saved versions before computing a multi-seed summary. Changing `RUN_SEEDS` does not alter the external dataset.


In [ ]:
def launch(architecture, seed, device):
    destination = OUTPUT / f'seed_{seed}' / architecture
    if (destination / 'run_report.json').exists():
        raise RuntimeError(f'Results already exist at {destination}; use a fresh Kaggle session or archive them')
    destination.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', 'scripts/run_original_kladb_external.py', '--architecture', architecture, '--seed', str(seed), '--device', device, '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--external-csv', str(EXTERNAL_CSV), '--external-manifest', str(EXTERNAL_MANIFEST), '--output-dir', str(destination)]
    log = (destination / 'training.log').open('w')
    return subprocess.Popen(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT), log, destination

if RUN_TRAINING:
    for seed in RUN_SEEDS:
        jobs = {}
        try:
            for architecture, device in (('baseline', 'cuda:0'), ('token_gated', 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0')):
                jobs[architecture] = launch(architecture, seed, device)
                if torch.cuda.device_count() == 1:
                    process, log, destination = jobs.pop(architecture)
                    code = process.wait()
                    log.close()
                    if code:
                        raise RuntimeError(f'{architecture} seed {seed} failed; inspect {destination / "training.log"}')
            while any(process.poll() is None for process, _, _ in jobs.values()):
                time.sleep(60)
                for architecture, (process, _, destination) in jobs.items():
                    lines = (destination / 'training.log').read_text(errors='replace').splitlines()
                    print(f'seed {seed}, {architecture}:', lines[-1] if lines else 'starting', flush=True)
            failures = []
            for architecture, (process, log, destination) in jobs.items():
                code = process.wait()
                log.close()
                if code:
                    failures.append(f'{architecture} (exit {code}; {destination / "training.log"})')
            jobs.clear()
            if failures:
                raise RuntimeError('Training failed: ' + ', '.join(failures))
            print('Completed matched seed', seed)
        finally:
            for process, log, _ in jobs.values():
                if process.poll() is None:
                    process.terminate()
                    process.wait()
                if not log.closed:
                    log.close()
else:
    print('Training disabled; set RUN_TRAINING = True after reviewing the preflight')


## 5. Generated results and integrity checks

The table below is built only from completed run reports. The prediction file must contain all 5,095 held-out sites with finite probabilities. No published or expected metric values are inserted. A mean/std table is shown only when all three declared paired seeds are present in this Kaggle output directory.


In [ ]:
import pandas as pd
import numpy as np
locked_sites = pd.read_csv(EXTERNAL_CSV)
locked_names = locked_sites['accession'].astype(str) + ':' + locked_sites['position_1based'].astype(str)
records = []
for seed in DECLARED_SEEDS:
    for architecture in ('baseline', 'token_gated'):
        destination = OUTPUT / f'seed_{seed}' / architecture
        report_file = destination / 'run_report.json'
        if not report_file.is_file():
            continue
        report = json.loads(report_file.read_text())
        if report['seed'] != seed or report['architecture'] != architecture or report['git_commit'] != COMMIT or report['preflight']['external_sha256'] != manifest['sha256']:
            raise ValueError(f'Result provenance mismatch for {architecture}, seed {seed}')
        predictions = pd.read_csv(destination / 'external_predictions.csv')
        if len(predictions) != 5095 or not predictions['name'].astype(str).equals(locked_names) or not predictions['true_label'].astype(int).equals(locked_sites['label'].astype(int)) or not predictions['window_51aa'].astype(str).equals(locked_sites['window_51aa'].astype(str)) or not np.isfinite(predictions['score']).all() or not predictions['score'].between(0, 1).all():
            raise ValueError(f'Invalid external predictions for {architecture}, seed {seed}')
        records.append({'architecture': architecture, 'seed': seed, 'selected_epoch': report['selected_epoch'], **report['external_metrics']})
results = pd.DataFrame.from_records(records)
if results.empty:
    print('No completed external evaluations in this Kaggle output directory')
else:
    display(results)
    results.to_csv(OUTPUT / 'external_results.csv', index=False)
    expected = {(seed, architecture) for seed in DECLARED_SEEDS for architecture in ('baseline', 'token_gated')}
    if set(zip(results['seed'], results['architecture'])) == expected:
        summary = results.groupby('architecture')[['AUPRC', 'AUC', 'MCC', 'F1', 'Pre', 'Rec', 'SP', 'ACC']].agg(['mean', 'std'])
        display(summary)
        summary.to_csv(OUTPUT / 'external_summary_three_seeds.csv')
    else:
        print('Three-seed paired summary pending; combine output bundles before calculating it')


## Interpretation boundary

Compare AUPRC against the external positive prevalence (254/5,095), and report AUC, MCC, sensitivity, and specificity with the stated 0.5 threshold. This protocol may yield lower scores than the original PCBert-Kla balanced test, which had substantial exact-window overlap with training. The old training duplicates remain a limitation, and the candidate-protein homology screen does not prove every original record's parent identity. Because this KlaDB holdout has already been examined in earlier project work, report this evaluation as **previously inspected external-study data**, not a fresh blinded confirmatory test. Further model changes would require a new untouched test for a confirmatory claim.
